# Session 10: End-to-End Machine Learning Pipeline & Deployment

Welcome to the closing session of our Data Science Masterclass! In this session, we will tie together everything we have learned about Python, Pandas, Scikit-Learn, and Decision Tree Ensembles. We will build a complete machine learning solution from raw data to a deployed web application.

## 🎯 Objectives
1. **Data Wrangling:** Clean real-world data with raw string units and missing values.
2. **Feature Engineering:** Extract temporal and categorical features from raw text.
3. **Robust Pipelines:** Build a `scikit-learn` `ColumnTransformer` and `Pipeline` to prevent data leakage and simplify deployment.
4. **Algorithm Comparison (The Battle):** Compare Linear Regression, Random Forest, and XGBoost.
5. **Streamlit Deployment:** Build an interactive web app and expose it from Google Colab using Localtunnel.

## 📊 About the Dataset: CarDekho Used Cars
We will use a popular Kaggle dataset containing specifications and selling prices of used cars.

### Feature Description:
* **name:** Brand and model of the car (e.g., "Maruti Swift Dzire VDI").
* **year:** Year of manufacture.
* **selling_price:** Price at which the car is being sold (Target variable).
* **km_driven:** Total kilometers driven by the car.
* **fuel:** Fuel type (Petrol, Diesel, LPG, CNG).
* **seller_type:** Type of seller (Individual, Dealer, Trustmark Dealer).
* **transmission:** Gear transmission (Manual, Automatic).
* **mileage:** Fuel economy (e.g., "23.4 kmpl" or "18.0 km/kg").
* **engine:** Engine displacement (e.g., "1248 cc").
* **max_power:** Maximum power output (e.g., "74 bhp" or "83.8 bhp").
* **seats:** Number of seats in the car.

## 📚 Learning References & Documentation
Before diving in, here are some reference materials to bookmark:
* **Pandas String Manipulation:** [Pandas User Guide on Text Data](https://pandas.pydata.org/pandas-docs/stable/user_guide/text.html)
* **Scikit-Learn ColumnTransformer:** [ColumnTransformer API](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html)
* **Scikit-Learn Pipelines:** [Pipeline API](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
* **XGBoost Regression:** [XGBoost Regressor API](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBRegressor)
* **Streamlit Documentation:** [Streamlit Get Started](https://docs.streamlit.io/)
* **Localtunnel:** [Localtunnel GitHub Repo](https://github.com/localtunnel/localtunnel)

In [ ]:
# Install XGBoost and Streamlit if they aren't already available
!pip install -q xgboost streamlit

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

print("Imports completed successfully!")

In [ ]:
# Load dataset directly from public raw GitHub URL
url = "https://raw.githubusercontent.com/manishkr1754/CarDekho_Used_Car_Price_Prediction/main/notebooks/data/cardekho_dataset.csv"
df = pd.read_csv(url)

print(f"Dataset shape: {df.shape}")
df.head()

## 🧪 Simulating Real-World Messy Data
The dataset we just loaded happens to be very clean. In reality, data extracted from databases or scraped from websites usually contains units (like 'kmpl' or 'cc') attached to the numbers, making them string types instead of numeric types. 

To make our data wrangling practice realistic, let's artificially add these units back into our dataframe and introduce some messy formatting!

In [ ]:
# Simulate messy string formats with units
df['mileage'] = df['mileage'].astype(str) + ' kmpl'
df['engine'] = df['engine'].astype(str) + ' cc'
df['max_power'] = df['max_power'].astype(str) + ' bhp'

# Revert pre-processed columns back to raw state so we can practice cleaning them!
df['year'] = 2026 - df['vehicle_age']
df['name'] = df['car_name']
df['fuel'] = df['fuel_type']
df['transmission'] = df['transmission_type']
df = df.drop(columns=['vehicle_age', 'brand', 'model', 'car_name', 'fuel_type', 'transmission_type', 'Unnamed: 0'])

# Introduce some empty/blank strings to simulate bad data entry
df.loc[0:20, 'max_power'] = ' bhp' 
df.loc[50:60, 'max_power'] = '  ' 

# Add a fake torque column to drop later
df['torque'] = '250Nm@ 1500-2500rpm'

print("Data is now successfully messy and ready for wrangling!")
df[['name', 'year', 'mileage', 'engine', 'max_power', 'torque']].head()

## 🛠️ Step 1: Data Wrangling & Cleaning
We face three major challenges:
1. **Mileage, Engine, and Max Power are objects (strings)** because they contain units (e.g., `kmpl`, `cc`, `bhp`). We need to extract the numbers and cast them to floats.
2. **Missing values and white spaces:** `max_power` contains blank strings like `" bhp"` or empty spaces which must be handled.
3. **Torque column:** Contains complex formats (e.g., `250Nm@ 1500-2500rpm`). In this session, we will drop it for simplicity.

In [ ]:
# Clean 'mileage': Remove units and convert to float
df['mileage'] = df['mileage'].str.replace(' kmpl', '', regex=False).str.replace(' km/kg', '', regex=False).astype(float)

# Clean 'engine': Remove ' cc' and convert to float
df['engine'] = df['engine'].str.replace(' cc', '', regex=False).astype(float)

# Clean 'max_power': Remove ' bhp', strip whitespaces, convert to numeric
df['max_power'] = df['max_power'].str.replace(' bhp', '', regex=False).str.strip()
df['max_power'] = pd.to_numeric(df['max_power'], errors='coerce') # Converts empty text/spaces to NaN

# Drop columns that are too noisy or redundant
df = df.drop(columns=['torque'])

# Check for remaining missing values
print(df.isnull().sum())
df.head()

## 🚗 Step 2: Feature Engineering
Let's engineer two key features:
1. **car_age:** Calculate how old the car is based on the year of manufacture. We will use `2026` as our current reference year.
2. **brand:** Extract the brand name (the first word from the `name` column).
3. **Rare category grouping:** Categorical features with too many unique values (high cardinality) can blow up our one-hot encoding. Let's group brands with fewer than 10 entries into a general `"Other"` category.

In [ ]:
# 1. Calculate Car Age
df['car_age'] = 2026 - df['year']

# 2. Extract Brand from Name
df['brand'] = df['name'].str.split(' ').str[0]

# 3. Group rare brands into 'Other'
brand_counts = df['brand'].value_counts()
rare_brands = brand_counts[brand_counts < 10].index
df['brand'] = df['brand'].replace(rare_brands, 'Other')

# Drop helper columns no longer needed
df = df.drop(columns=['name', 'year'])

print("Unique brands after grouping:", df['brand'].nunique())
df.head()

## ⚙️ Step 3: Train-Test Split & Preprocessing Pipeline
To prevent **data leakage**, we must split our data into training and testing sets before setting up any imputation or scaling. 

We will build a clean `ColumnTransformer` to automate the preprocessing steps for:
* **Numerical Features:** Impute missing values with the median, then scale.
* **Categorical Features:** Impute missing values with the most frequent category, then apply One-Hot Encoding.

In [ ]:
# Split features and target
X = df.drop(columns=['selling_price'])
y = df['selling_price']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define column names
num_features = ['km_driven', 'mileage', 'engine', 'max_power', 'seats', 'car_age']
cat_features = ['fuel', 'seller_type', 'transmission', 'brand']

# Preprocessing pipelines
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine preprocessors
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

print("Preprocessing Pipeline Defined!")

## ⚔️ Step 4: The Algorithm Battle (Linear vs. Random Forest vs. XGBoost)
We will now train three different regressors inside our full pipeline:
1. **Linear Regression:** An easily interpretable baseline model.
2. **Random Forest Regressor:** An ensemble method based on bagging decision trees.
3. **XGBoost Regressor:** A state-of-the-art gradient boosting framework.

In [ ]:
# Define models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42)
}

# Run the battle
for name, model in models.items():
    # Build complete pipeline (preprocess + model)
    full_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # Train
    full_pipeline.fit(X_train, y_train)
    
    # Predict
    preds = full_pipeline.predict(X_test)
    
    # Evaluate
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    print(f"=== {name} ===")
    print(f"Mean Absolute Error (MAE): ₹ {mae:,.2f}")
    print(f"R² Score: {r2:.4f}\n")

## 💾 Step 5: Exporting the Champion Model
Let's select the best-performing model (typically Random Forest or XGBoost) and save the entire pipeline object. This ensures that when we load it in deployment, it will handle raw user input transformations automatically!

In [ ]:
# Retrain the champion model pipeline
champion_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(n_estimators=100, random_state=42))
])

champion_pipeline.fit(X_train, y_train)

# Save pipeline as a binary file
joblib.dump(champion_pipeline, 'car_price_pipeline.pkl')
print("Model pipeline exported as 'car_price_pipeline.pkl'!")

## 💻 Step 6: Create the Streamlit Web Application
Now we will write our web app code to a file named `app.py` directly from Colab using the `%%writefile` magic command.

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import numpy as np

# Load the trained model pipeline
pipeline = joblib.load('car_price_pipeline.pkl')

st.set_page_config(page_title="Used Car Price Predictor", layout="centered", page_icon="🚗")
st.title("🚗 Used Car Price Predictor")
st.write("Predict the resale value of a used car instantly based on its specifications.")

# Setup two-column layout for input fields
col1, col2 = st.columns(2)

with col1:
    km_driven = st.number_input("Kilometers Driven", min_value=0, value=50000, step=1000)
    mileage = st.number_input("Mileage (kmpl)", min_value=0.0, value=18.0, step=0.5)
    engine = st.number_input("Engine Capacity (cc)", min_value=500, max_value=8000, value=1200, step=100)
    max_power = st.number_input("Max Power (bhp)", min_value=10.0, max_value=1000.0, value=85.0, step=5.0)
    seats = st.slider("Seats", min_value=2, max_value=10, value=5)
    car_age = st.slider("Car Age (Years)", min_value=0, max_value=30, value=5)

with col2:
    fuel = st.selectbox("Fuel Type", ['Petrol', 'Diesel', 'CNG', 'LPG', 'Electric'])
    seller_type = st.selectbox("Seller Type", ['Dealer', 'Individual', 'Trustmark Dealer'])
    transmission = st.selectbox("Transmission", ['Manual', 'Automatic'])
    brand = st.selectbox("Brand", ['Maruti', 'Hyundai', 'Honda', 'Toyota', 'Ford', 'Chevrolet', 'Volkswagen', 'Renault', 'Other'])

# Run prediction when user clicks the button
if st.button("Predict Resale Price", type="primary"):
    # Structure input features exactly as in the training dataset
    input_data = pd.DataFrame({
        'km_driven': [km_driven],
        'mileage': [mileage],
        'engine': [engine],
        'max_power': [max_power],
        'seats': [seats],
        'car_age': [car_age],
        'fuel': [fuel],
        'seller_type': [seller_type],
        'transmission': [transmission],
        'brand': [brand]
    })
    
    # Run prediction through our model pipeline
    prediction = pipeline.predict(input_data)[0]
    
    # Display the result formatted nicely
    st.success(f"### 💰 Estimated Resale Value: ₹ {prediction:,.2f}")

## 🚀 Step 7: Local Deployment and Exposing the Tunnel
In Colab, we run our Streamlit server in the background and expose it using Localtunnel.

### Instructions:
1. Copy the **IP Address** outputted by the second code block. This is your tunnel password.
2. Click the **Localtunnel URL** generated at the bottom.
3. Paste the IP address in the security landing page, and click submit. Enjoy your live web app!

In [ ]:
# 1. Install localtunnel globally
!npm install -g localtunnel

# 2. Get the Colab instance's public IP address (used as the localtunnel entry password)
print("\n--- Tunnel Details ---")
print("STEP 1: Copy this IP Address as your password:")
!curl ipv4.icanhazip.com

# 3. Start Streamlit in the background
import subprocess
print("\nSTEP 2: Starting Streamlit...")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# 4. Open the tunnel
print("STEP 3: Click the URL below to launch your Web App:")
!npx localtunnel --port 8501